# 1. Librerías y funciones

In [6]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None 
from sklearn import metrics
import pycaret
import mlflow

In [7]:
import sbp_modelo as sbp

# 2. Importando data

In [8]:
PROCESSED_DATA_DIR = "../data/processed/"
MLRUNS_DIR = "../mlruns/"
LOGS_DIR = "../logs/"
MODELS_DIR = "../models/"

In [9]:
# Importando tabla
df_train = pd.read_parquet(PROCESSED_DATA_DIR+"train.parquet")
df_train.shape

(4629620, 8)

In [10]:
df_train.head(5)

,type,amount,entTrxType,oldbalanceOrg,origTrxAmount,oldbalanceDest,destTrxAmount,isFraud
5508643,CASH_OUT,19951.73,CC,49660.00,-19951.73,0.00,19951.73,0
2321526,PAYMENT,6357.25,CM,56503.00,-6357.25,0.00,NaN,0
4111006,CASH_IN,107883.86,CC,3488609.81,107883.86,885589.22,-107883.86,0
2716783,PAYMENT,6514.55,CM,31372.00,-6514.55,0.00,NaN,0
2137572,PAYMENT,2422.66,CM,0.00,0.00,0.00,NaN,0


---

# 3. Auto ML

In [11]:
data = df_train.copy()
target = 'isFraud'

In [12]:
data.dtypes

type              category
amount             float64
entTrxType        category
oldbalanceOrg      float64
origTrxAmount      float64
oldbalanceDest     float64
destTrxAmount      float64
isFraud              int64
dtype: object

In [13]:
# MLflow
mlflow.set_tracking_uri(MLRUNS_DIR)

In [14]:
from pycaret.classification import *
from sklearn.metrics import average_precision_score

s = setup(data=data, target=target, session_id=666, normalize=True, fold=5, n_jobs=-1, numeric_imputation=0,
         log_experiment = True, experiment_name = 'fraude_automl', log_plots = False, log_data = False,
         system_log = LOGS_DIR+"pycaret.log")

add_metric('pr_auc', 'PR-AUC', average_precision_score, target='pred_proba')

,Description,Value
0,Session id,666
1,Target,isFraud
2,Target type,Binary
3,Original data shape,"(4629620, 8)"
4,Transformed data shape,"(4629620, 12)"
5,Transformed train set shape,"(3240734, 12)"
6,Transformed test set shape,"(1388886, 12)"
7,Numeric features,5
8,Categorical features,2
9,Rows with missing values,33.7%


Name                                                            PR-AUC
Display Name                                                    PR-AUC
Score Function       <pycaret.internal.metrics.EncodedDecodedLabels...
Scorer               make_scorer(average_precision_score, response_...
Target                                                      pred_proba
Args                                                                {}
Greater is Better                                                 True
Multiclass                                                        True
Custom                                                            True
Name: pr_auc, dtype: object

In [15]:
top = compare_models(sort='F1', n_select=5, include=['rf','lightgbm','xgboost','catboost','dt','lr','knn','gbc'])

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,PR-AUC,TT (Sec)
catboost,CatBoost Classifier,0.9997,0.9985,0.7277,0.9273,0.8150,0.8149,0.8211,0.8636,193.3260
knn,K Neighbors Classifier,0.9997,0.8941,0.7022,0.9028,0.7896,0.7895,0.7959,0.7198,546.5400
dt,Decision Tree Classifier,0.9997,0.8903,0.7808,0.7806,0.7806,0.7804,0.7805,0.6099,11.0280
xgboost,Extreme Gradient Boosting,0.9995,0.8277,0.5365,0.6922,0.6029,0.6026,0.6083,0.4865,25.4080
lr,Logistic Regression,0.9995,0.9850,0.3964,0.8493,0.5403,0.5401,0.5799,0.5317,4.4260
gbc,Gradient Boosting Classifier,0.9992,0.6765,0.1669,0.5323,0.2261,0.2258,0.2756,0.1327,255.6800


2026/10/07 18:06:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/07 19:14:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/07 19:15:45 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/07 19:16:27 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/07 19:16:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/07 19:16:57 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be adde

In [18]:
# Se selecciona Catboost por ser top F1
top_model = top[0]
top_model

CatBoostClassifier(border_count=254, random_state=666, task_type='CPU', verbose=False)

In [19]:
save_model(top_model, MODELS_DIR)

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['amount', 'oldbalanceOrg',
                                              'origTrxAmount', 'oldbalanceDest',
                                              'destTrxAmount'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill_value=0,
                                                               keep_empty_features=False,
                                                               missing_values=nan,
                                                               strategy='constant'))),
                 ('categorical_imputer',
                  Transfor...
                                                               drop_invariant=False,
 